In [48]:
# config.py

#--------Packages
from pathlib import Path

### Below from SMM notes
import pandas as pd
import matplotlib.pyplot as plt
import dask
from dask import delayed, compute
from dask.distributed import Client
import scipy.optimize as optimize
from scipy.optimize import minimize
import scipy.stats as stats
import plotly.express as px
import numpy as np

!{sys.executable} -m pip install quantecon
import quantecon as qe
from quantecon.markov.approximation import tauchen

#---------relative path configuration

# Root = folder containing config.py

from pathlib import Path

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd()

SCRIPTS = ROOT / "scripts"
RESULTS = ROOT / "results"

RESULTS.mkdir(exist_ok=True)

print("Setup successful")
print(f"ROOT: {ROOT}")
print(f"SCRIPTS: {SCRIPTS}")
print(f"RESULTS: {RESULTS}")



zsh:1: parse error near `-m'
Setup successful
ROOT: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming
SCRIPTS: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/scripts
RESULTS: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/results


In [49]:
# Step 1: Define the model parameters

#--------CE Calibrated Parameters

beta = 0.95      # discount factor
delta = 0.15     # depreciation rate
p = 1.0          # price of capital

#--------SMM starting parameters

alpha = 0.70
gamma = 0.13
rho = 0.10
sigma = 0.89
phi_tilde0 = 0.0


#Psi d ---

target_moments = [
    0.03,   # a1
    0.24,   # a2
    0.40,   # serial correlation of I/K
    0.25,   # std(pi/K)
    3.00,   # average Q
    0.25    # fraction externally financed
]

weighting_matrix = np.eye(len(target_moments))  # Identity matrix for simplicity



In [50]:
import numpy as np

# 1. Setup Grids and Parameters
K_grid = np.linspace(1, 100, 100)  # Capital grid
A_grid = np.array([0.9, 1.0, 1.1])  # Productivity grid (e.g., Markov chain states)
Pi_prob = np.array(
    [[0.7, 0.2, 0.1], [0.15, 0.7, 0.15], [0.1, 0.2, 0.7]]
)  # Transition matrix for A

beta = 0.95
delta = 0.10
p = 1.0
phi0 = 2.0
phi1 = 0.15

# Pre-allocate Value Functions: shape (num_K, num_A)
V = np.zeros((len(K_grid), len(A_grid)))
V_next = np.zeros_like(V)
policy_K = np.zeros_like(V)

# Precompute deterministic parts for speed if possible, or loop
max_iter = 500
tol = 1e-6

# 2. Main VFI Loop
for it in range(max_iter):
    # Compute expected continuation value: shape (num_K, num_A)
    # E[V(K', A') | A] = V(K', A') * Pi_prob^T
    EV = V @ Pi_prob.T

    for i_A, A in enumerate(A_grid):
        for i_K, K in enumerate(K_grid):

            # Current period profit
            pi_KA = A * (K**0.3)  # Example profit function

            # Internal funds available today
            internal_funds = pi_KA + (1 - delta) * K

            # Evaluate ALL possible K' choices from the grid
            # Vectorized over all possible next-period capital choices
            K_prime = K_grid

            # 3. Calculate value for each choice of K'
            # Check internal vs external financing for every K'
            is_external = K_prime > internal_funds

            # Cost and penalty calculations
            investment = K_prime - (1 - delta) * K
            C_K_Kp = 0.0  # Insert adjustment cost function if applicable

            # External vs Internal payoffs
            payoff = np.zeros_like(K_prime)

            # External Financing Case
            payoff[is_external] = (
                pi_KA
                - p * investment[is_external]
                - C_K_Kp
                - phi0
                - phi1 * (p * investment[is_external] - pi_KA)
            )

            # Internal Financing Case
            payoff[~is_external] = pi_KA - p * investment[~is_external] - C_K_Kp

            # Total Value = Current Payoff + Beta * Expected Continuation Value
            # EV has shape (num_Kp, num_A), we need EV for all K_prime at state i_A
            total_value = payoff + beta * EV[:, i_A]

            # 4. The Max Operator (takes the upper envelope automatically)
            V_next[i_K, i_A] = np.max(total_value)
            policy_K[i_K, i_A] = K_grid[np.argmax(total_value)]

    # Check convergence
    if np.max(np.abs(V_next - V)) < tol:
        print(f"Converged in {it} iterations.")
        V = V_next.copy()
        break

    V = V_next.copy()


Converged in 270 iterations.


In [51]:
# Step 2: 

"""Simulate the model to generate data on investment, Q, and cash flow from the simulated panel of firms.

#Per CE, utilize the optimization problem (eq. 1), adjustment function (eq. 4), and the profit function (eq. 8)

#Use resulting policy function to create a panel data set.

#Psi S is obtained from here.

Secondary:
#Per assignment,  generate data on investment, Q, and cash flow.


#GOAL is to derive policy function: K' = h(K,A)

#Basics - Define functions 1, 4, and 8 and create grids for A and K.

#Objective function (Dynamic investment): 


#max (profit - investment - adjustment costs + beta * expected future value)

"""

#Define functions

# 1. Profit function 
def profit(K, A, alpha):
    return A * K**alpha

# 2. Investment function 
def investment(K, K_next, delta):
    return K_next - (1 - delta) * K

# 3. Adjustment costs now use GROSS investment from Equation (4)
def adjustment_costs(K, K_next, delta, gamma):
    I = investment(K, K_next, delta)
    return (gamma / 2) * (I / K) ** 2 * K


# Calculate current period net cash flow
def current_payout(K, A, alpha, K_next, delta, gamma):
    return (
        profit(K, A, alpha)
        - investment(K, K_next, delta)
        - adjustment_costs(K, K_next, delta, gamma)
    )


In [52]:
# =============================================================================
# 2. GRID GENERATION FUNCTIONS
# =============================================================================
def make_A_process(rho, sigma, nA=5):
    # QuantEcon tauchen method wrapper
    mc = tauchen(n=nA, rho=rho, sigma=sigma)
    logA_grid = np.asarray(mc.state_values)
    P = np.asarray(mc.P)

    # Ensure transition probabilities are valid
    P = np.where(P < 0, 0.0, P)
    P = P / P.sum(axis=1, keepdims=True)
    A_grid = np.exp(logA_grid)

    return A_grid, P

def make_K_grid(K_min=1.0, K_max=1000.0, nK=300):
    return np.linspace(K_min, K_max, nK)


# =============================================================================
# 3. DYNAMIC PROGRAMMING ENGINE
# =============================================================================
def CooperEjarqueModel(
    K_grid,
    A_grid,
    P,
    alpha,
    gamma,
    delta,
    beta,
    phi0,
    phi1=0.0,
    v_tol=1e-6,
    max_iter=1000
):
    nK = len(K_grid)
    nA = len(A_grid)

    # Initialize array containers
    V = np.zeros((nK, nA))          
    policy = np.zeros((nK, nA), dtype=int)  
    
    # Track financing: 0 = internal, 1 = external equity issued (negative payout)
    finance_policy = np.zeros((nK, nA), dtype=int)

    V_dist = 10.0
    iteration = 0

    print("Starting Value Function Iteration...")

    while (V_dist > v_tol) and (iteration < max_iter):
        V_new = np.zeros_like(V)
        # Loop over today's states
        for iK, K in enumerate(K_grid):
            for iA, A in enumerate(A_grid):
                
                # Vector to hold outcomes for every possible choice of next-period capital
                values = np.full(nK, -np.inf)
                temp_finance = np.zeros(nK, dtype=int)
                
                # Loop over tomorrow's choices
                for iKp, K_next in enumerate(K_grid):
                    
                   # 1. Base payout before frictions
                    net_funds = (
                        profit(K, A, alpha) 
                        - investment(K, K_next, delta) 
                        - adjustment_costs(K, K_next, delta, gamma)
                    )
                    
                    # =========================================================
                    # 2. EXTERNAL VS INTERNAL FINANCING (MATCHING EQUATION 11)
                    # =========================================================
                    # Condition from image: K' > pi(K,A) + (1-delta)K
                    if K_next > (profit(K, A, alpha) + (1 - delta) * K):
                        
                        # External funding penalty: phi0 + phi1 * (p*I - pi)
                        # (Assuming price p = 1.0 based on your investment function)
                        I = investment(K, K_next, delta)
                        pi = profit(K, A, alpha)
                        
                        equity_issuance_cost = phi0 + phi1 * (I - pi)
                        payout = net_funds - equity_issuance_cost
                        temp_finance[iKp] = 1  # External funding
                        
                    else:
                        # Internal funding (No penalties applied)
                        payout <= net_funds
                        temp_finance[iKp] = 0  # Internal funding
                    
                # Find and store the absolute best choice of K_next
                best_iKp = np.argmax(values)
                V_new[iK, iA] = values[best_iKp]
                policy[iK, iA] = best_iKp
                finance_policy[iK, iA] = temp_finance[best_iKp]
                
        # Calculate convergence distance
        V_dist = np.max(np.abs(V_new - V))
        V = np.copy(V_new)
        iteration += 1

    print(f"Converged successfully in {iteration} iterations!")
    print(f"Final V distance metric: {V_dist:.8f}\n")

    return V, policy, finance_policy


In [53]:
# Simulate firms using dask


@delayed
def simulate_single_firm(policy_indices, finance_policy, K_grid, A_grid, P_cum, alpha, delta, num_periods, burn_in):
    """
    Simulates a single firm's lifecycle trajectory.
    """
    T = num_periods - burn_in
    
    firm_K = np.zeros(T)
    firm_I = np.zeros(T)
    firm_A = np.zeros(T)
    firm_fin = np.zeros(T)
    firm_prof = np.zeros(T)

    # Initialize at a neutral midpoint
    iK = len(K_grid) // 2
    iA = len(A_grid) // 2

    for t in range(num_periods):
        iKp = policy_indices[iK, iA]
        fin_type = finance_policy[iK, iA]
        
        K_today = K_grid[iK]
        K_next = K_grid[iKp]
        A_today = A_grid[iA]
        
        prof_today = A_today * (K_today ** alpha)
        inv_today = K_next - (1 - delta) * K_today

        # Throw away burn-in periods, only store steady-state data
        if t >= burn_in:
            idx = t - burn_in
            firm_K[idx] = K_today
            firm_I[idx] = inv_today
            firm_A[idx] = A_today
            firm_fin[idx] = fin_type
            firm_prof[idx] = prof_today

        iK = iKp
        r = np.random.rand()
        iA = np.searchsorted(P_cum[iA, :], r)

    return firm_K, firm_I, firm_fin, firm_prof


def run_distributed_panel_simulation(policy_indices, finance_policy, K_grid, A_grid, P, num_firms=5000, num_periods=150, burn_in=30, alpha=0.33, delta=0.10):
    """
    Spins up a local cluster client to compute panel paths in parallel.
    """
    # 1. Initialize the distributed cluster client
    # This automatically opens a local dashboard to track cluster activity
    client = Client()
    print(f"Distributed dashboard available at: {client.dashboard_link}")
    
    P_cum = np.cumsum(P, axis=1)
    
    # 2. Build the task graph using delayed
    tasks = []
    for _ in range(num_firms):
        task = simulate_single_firm(
            policy_indices, finance_policy, K_grid, A_grid, P_cum, alpha, delta, num_periods, burn_in
        )
        tasks.append(task)
        
    print(f"Submitting graph to cluster for {num_firms} parallel firm tasks...")
    
    # 3. Execute the distributed compute step across workers
    results = compute(*tasks)
    
    # Close the cluster worker connections neatly
    client.close()
    
    # 4. Unpack collection structures into shape (num_firms, T) matrices
    sim_K = np.array([r[0] for r in results])
    sim_I = np.array([r[1] for r in results])
    sim_fin = np.array([r[2] for r in results])
    sim_prof = np.array([r[3] for r in results])
    
    # 5. Calculate summary statistics for table replication
    ik_ratio = sim_I / sim_K
    mean_IK = np.mean(ik_ratio)
    std_IK = np.std(ik_ratio)
    equity_freq = np.mean(sim_fin) * 100 
    corr_I_Prof = np.corrcoef(sim_I.flatten(), sim_prof.flatten())[0, 1]

    print("\n" + "="*40)
    print(" DISTRIBUTED DASK SIMULATION RESULTS   ")
    print("="*40)
    print(f"Mean Investment Rate (I/K):    {mean_IK:.4f}")
    print(f"Std Dev Investment Rate:       {std_IK:.4f}")
    print(f"Equity Issuance Frequency (%): {equity_freq:.2f}%")
    print(f"Corr(Investment, Profit):      {corr_I_Prof:.4f}")
    print("="*40)

    return sim_K, sim_I, sim_fin, sim_prof


In [55]:
def smm_objective(theta, K_min, K_max, nK, nA, delta, beta, phi1):
    """
    Computes the structural distance metric between simulated and data moments.
    """
    alpha, gamma, rho, sigma, phi0 = theta
    
    # >>> ADD THIS PRINT TO TRACK IMMEDATE ACTIVITY = 1 or gamma < 0 or rho <= 0 or rho >= 1 or sigma <= 0 or phi0 < 0:
    if alpha <= 0 or alpha >= 1 or gamma < 0 or rho <= 0 or rho >= 1 or sigma <= 0 or phi0 < 0:
        print("--> Invalid parameter boundaries encountered. Applying penalty.", flush=True)
        return 1e10

        
    # Step A: Re-generate the state space grids and Markov matrix
    A_grid, P = make_A_process(rho, sigma, nA=nA)
    K_grid = make_K_grid(K_min, K_max, nK=nK)
    
    # Step B: Solve the firm's model
    V, policy_indices, finance_policy = CooperEjarqueModel(
        K_grid, A_grid, P, alpha, gamma, delta, beta, phi0, phi1, max_iter=400, v_tol=1e-5
    )
    
    # Step C: Generate parallel panel paths using the distributed Dask simulator
    P_cum = np.cumsum(P, axis=1)
    tasks = []
    for _ in range(2000):  
        task = simulate_single_firm(
            policy_indices, finance_policy, K_grid, A_grid, P_cum, alpha, delta, num_periods=100, burn_in=30
        )
        tasks.append(task)
        
    results = compute(*tasks)
    
    # =============================================================================
    # >>> FIXED UNPACKING HERE: Extracting specific elements from the tuple <<<
    # =============================================================================
    sim_K = np.array([r[0] for r in results])
    sim_I = np.array([r[1] for r in results])
    sim_fin = np.array([r[2] for r in results])
    sim_prof = np.array([r[3] for r in results])
    
    # Step E: Calculate the 4 matching model moments
    ik_ratio = sim_I / sim_K
    model_mean_IK = np.mean(ik_ratio)
    model_std_IK = np.std(ik_ratio)
    model_equity_freq = np.mean(sim_fin) * 100
    model_mean_prof = np.mean(sim_prof)
    
    simulated_moments = np.array([
        model_mean_IK, 
        model_std_IK, 
        model_equity_freq, 
        model_mean_prof
    ])
    
    # Step F: Calculate the SMM loss score
    moment_deviation = simulated_moments - target_moments
    smm_loss = moment_deviation @ weighting_matrix @ moment_deviation.T
    
    print(f"Current Parameters -> alpha: {alpha:.3f}, gamma: {gamma:.3f}, rho: {rho:.3f}, sigma: {sigma:.3f}, phi0: {phi0:.3f} | Loss: {smm_loss:.6f}")
    return smm_loss